### Middleware 
Middleware provides a way to more tightly control what happens inside the agent . Middleware is useful for the following:
* Tracking agent behavior with logging , analytics and debugging 
* Tranforming prompts , tool selection and output formatting 
* adding retries, fallbacks and early termination logic 
* Applying rate limits,guardrails, and PII detection 

In [14]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### summarization Middleware
Automatically sumarize conversation history when approching token limits,preserving recent messages while compressing older context. Summarization is useful for the following. 
- Long-running conversations that exceed context windows.
- Multi - turn dialogues with extensive history.
- application where preserving full conversation context matters

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### message based summarization 
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [16]:
### Run with thread id 
config = {"configurable":{"thread_id":"test-1"}}

In [17]:
# Alternative  test data 
questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 15-7?",
    "what is 3*3?",
    "what is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Message: {response}")
    print(f"Message: {len(response['messages'])}")

Message: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='416173d5-750f-4f5c-9cbd-867381154f11'), AIMessage(content="<think>\nOkay, let's figure out what 2 plus 2 is. Hmm, so I know that addition is combining two numbers. Let me start with the basics. If I have two apples and someone gives me two more apples, how many do I have in total? Well, two apples plus two apples should make four apples. That seems right because 1 + 1 is 2, and adding another 1 + 1 would make 4. Wait, maybe I should count on my fingers. Let's see, 1, 2 on one hand, and 1, 2 on the other hand. Combining them all together, 1, 2, 3, 4. Yeah, that's four.\n\nAlternatively, in mathematics, the addition of numbers is straightforward. Each number is a quantity, and adding them together gives the total. So 2 is a quantity representing two units, and adding another two units would logically result in four units. I can also think about the number line. Starting at 2 and mo

### Token Size 

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool 
def search_hotels(city:str) -> str:
    """ Search hotels - returns long response to use more tokens."""
    return f""" Hotels in {city}
    1. Grand hotel - 5 star, $350/night, spa, pool,gym 
    2. City Inn - 4 star , $180/night ,business center
    3. Budget Stay - 3 star, $75/night, free wifi """

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}

# Token counter(approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars=  1 token

In [20]:
# Run test 
cities = ["Paris","London","tokyo","New York", "Dubai", "singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find Hotels in {city}")]},
        config = config
    )
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~ {tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~ 804 tokens, 8 messages
[HumanMessage(content="Here is a summary of the conversation to date:\n\n<think>\nOkay, let me tackle this. The user wants me to extract the highest quality context from their conversation history. First, I need to understand the main goal here. The session intent is about finding hotels in London after initially looking at Paris. The summary should capture the transition from Paris to London, the structure of the search results, and any pending actions. The artifacts include the two hotel searches, Paris and London. Next steps would involve presenting the London results, checking if the user needs more info, and handling any follow-up. I need to make sure each section is concise and only the most relevant info is included. Let me structure it step by step.\n\nWait, the user also provided a previous AI message where they listed the London hotels. Should that be part of the artifacts? Hmm, the artifacts are about created or modified resources. The AI's me

### Fraction 

In [23]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool 
def search_hotels(city:str) -> str:
    """ Search hotels - returns long response to use more tokens."""
    return f""" Hotels in {city}
    1. Grand hotel - 5 star, $350/night, spa, pool,gym 
    2. City Inn - 4 star , $180/night ,business center
    3. Budget Stay - 3 star, $75/night, free wifi """

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction",0.05), #  0.5 % = ~640 tokens
            keep=("fraction",0.02)    # 0.2 % of -256 token
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}

# Token counter(approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars=  1 token

# cities 
cities = ["Paris","London","tokyo","New York", "Dubai", "singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find Hotels in {city}")]},
        config = config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 # qwen context
    print(f"{city}: ~ {tokens} tokens({fraction:.4%}),  ,{len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~ 152 tokens(0.1187%),  ,4 messages
[HumanMessage(content='Find Hotels in Paris', additional_kwargs={}, response_metadata={}, id='35896b56-2112-4cc4-ba9b-30e3a49bed6d'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking to find hotels in Paris. Let me check the available tools. There's a function called search_hotels that takes a city parameter. The description says it returns a long response to use more tokens, which might be important for the response length. I need to call this function with the city set to Paris. The parameters require the city as a string, so I'll structure the tool call accordingly. Just make sure the JSON is correctly formatted with the city name in the arguments.\n", 'tool_calls': [{'id': 'b7jk24f6s', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 155, 'total_tokens': 282, 'completion_time': 0

## human in the loop middelware 

pause agent exectuion for human approval, editing or rejection of tool calls before they execute. Human in the loop is useful for the following 
- High stakes operations requiring human approval(eg: Database writes, financial transcations)
- Compliance workflows where human oversight is mandatory 
- long running coversations where human feedback guides the agent

In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [25]:
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [26]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [27]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7af6abfd-83fb-47f2-81a1-0d24a80d3b76'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three: recipient is john@test.com, subject is Hello, body is How are you?. So I need to call send_email_tool with these arguments. No issues here. No need to use the read_email_tool since the user isn't asking to read an email. Just send it.\n", 'tool_calls': [{'id': 'xde6jhytd', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata=

In [28]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: The email has been successfully sent to john@test.com with the subject "Hello" and the message "How are you?". Let me know if you need further assistance!


In [29]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7af6abfd-83fb-47f2-81a1-0d24a80d3b76'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three: recipient is john@test.com, subject is Hello, body is How are you?. So I need to call send_email_tool with these arguments. No issues here. No need to use the read_email_tool since the user isn't asking to read an email. Just send it.\n", 'tool_calls': [{'id': 'xde6jhytd', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata=

Reject

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [31]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [33]:
# Step 2: Reject
if "__interrupt__" in result:
    print("Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

In [34]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='9659ecda-fe57-40c2-8cd1-ac162e10a1f8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information: recipient is john@test.com, subject is Hello, body is How are you?. So I need to call send_email_tool with those arguments. No need to use the read_email_tool here since the user isn't asking to read an email. Just construct the JSON object with the parameters and make the tool call.\n", 'tool_calls': [{'id': 'mr20fcnjy', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hell

Editing

In [37]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [38]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [39]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

Paused! Editing...
Result: The email has been successfully sent to **correct@email.com** with the corrected subject **"Corrected Subject"**. Let me know if you need further assistance!


In [40]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='5cf8920a-9a43-4247-9e78-6c8ac5966327'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'Test', and body is 'Hello'. I need to make sure all required fields are included. Yep, all three are required and present. I'll call the send_email_tool with these parameters.\n", 'tool_calls': [{'id': '7cw7gkkb4', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 249, 'total_tokens'